# Notebook 02 – Data Preprocessing & Feature Engineering

Prepare the raw dataset for machine learning training.

**What this notebook does:**
1. Loads the raw dataset (`all_career_1980_with_allstar.csv`)
2. Handles missing values in statistical columns
3. Encodes categorical features (team abbreviations)
4. Scales numerical features for model training
5. Addresses class imbalance using SMOTE
6. Saves the preprocessed dataset for training

**Output**: `data/preprocessed_features.csv` – ready for model training

In [2]:
import sys, subprocess
packages = [
    "numpy", "pandas", "scikit-learn", "imbalanced-learn", 
    "matplotlib", "seaborn", "joblib"
]
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Dependencies ready.")

Dependencies ready.


In [21]:
import sys
from pathlib import Path

# Make sure src/ is on the path when running from notebooks/
REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

    # ── Configuration ────────────────────────────────────────────────────────
DATA_DIR = REPO_ROOT / "data"
INPUT_FILE = DATA_DIR / "all_career_1980_with_allstar.csv"
OUTPUT_FILE = DATA_DIR / "preprocessed_features.csv"
SCALER_FILE = DATA_DIR / "scaler.pkl"

print(f"Input: {INPUT_FILE}")
print(f"Output: {OUTPUT_FILE}")

Input: C:\Users\Dongyoon Nam\Desktop\NBA_ALLSTAR_PREDICTION\data\all_career_1980_with_allstar.csv
Output: C:\Users\Dongyoon Nam\Desktop\NBA_ALLSTAR_PREDICTION\data\preprocessed_features.csv


## Step 1: Handle Missing Values

Fill missing values in statistical columns (GS, FG_PCT, FG3_PCT, FT_PCT):
- **Player-level average** if the player has other data points for that stat
- **Column-wide average** if the player has no data for that stat

In [22]:
# Check missing values
print("Missing values before handling:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Fill missing values: player average if available, otherwise column average
cols_to_fill = ['GS', 'FG_PCT', 'FG3_PCT', 'FT_PCT']

for col in cols_to_fill:
    if col in df.columns:
        # Calculate player-level average (groupby PLAYER_ID)
        player_avg = df.groupby('PLAYER_ID')[col].transform('mean')
        
        # Fill: use player average if available, otherwise use column average
        col_mean = df[col].mean()
        df[col] = df[col].fillna(player_avg).fillna(col_mean)
        
        print(f"  ✓ {col}: Filled with player average (if available) or column average")

print(f"\nMissing values after handling: {df.isnull().sum().sum()}")

Missing values before handling:
Series([], dtype: int64)
  ✓ GS: Filled with player average (if available) or column average
  ✓ FG_PCT: Filled with player average (if available) or column average
  ✓ FG3_PCT: Filled with player average (if available) or column average
  ✓ FT_PCT: Filled with player average (if available) or column average

Missing values after handling: 0


## Step 2: Feature Engineering

Encode categorical features and select numerical features for training.

In [23]:
# One-Hot Encoding for team
df_encoded = pd.get_dummies(df, columns=['team'], prefix='team')

# Drop identifier columns
feature_cols = [col for col in df_encoded.columns 
                if col not in ['PLAYER_ID', 'first', 'last', 'year', 'allstar']]

X = df_encoded[feature_cols]
y = df_encoded['allstar']

print(f"Features: {X.shape[1]} columns")
print(f"Samples: {len(y)} rows")

Features: 57 columns
Samples: 21892 rows


## Step 3: Feature Scaling

Standardize numerical features to zero mean and unit variance.

In [24]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Save scaler for future use
joblib.dump(scaler, SCALER_FILE)
print(f"Scaler saved to: {SCALER_FILE}")

Scaler saved to: C:\Users\Dongyoon Nam\Desktop\NBA_ALLSTAR_PREDICTION\data\scaler.pkl


## Step 4: Save Preprocessed Data

Save the final preprocessed dataset for model training.

In [33]:
# Create final dataframe
preprocessed_df = pd.DataFrame(X_scaled, columns=X.columns)
preprocessed_df["allstar"] = y.values

preprocessed_df.to_csv(OUTPUT_FILE, index=False)
print(f"\nPreprocessed dataset saved to: {OUTPUT_FILE}")
print(f"Shape: {preprocessed_df.shape}")
print(f"Ready for training!")


Preprocessed dataset saved to: C:\Users\Dongyoon Nam\Desktop\NBA_ALLSTAR_PREDICTION\data\preprocessed_features.csv
Shape: (21892, 58)
Ready for training!
